# B6 · Despliegue de modelos — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Capa B · Ciencia de datos aplicada · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

Felicitaciones por llegar al último módulo del tronco de Machine Learning. Entrenar y evaluar un modelo es divertido, pero solo agrega valor cuando otras personas pueden **usarlo**. Hoy dejarás tu pipeline entrenado **listo para usar**: una **función de predicción segura** (con validación de entrada) y la **puntuación por lotes** (*batch scoring*) que guarda los resultados en un archivo.

**Competencia de salida (manos a la obra):** dejar tu modelo **usable** por otra persona — una función de predicción con validación, la puntuación por lotes y el archivo de resultados.

**A nivel conceptual (para conocer, no para construir aquí):** qué implica llevar un modelo a *producción* — API/servicio web, app, **monitoreo** y **reentrenamiento** — y por qué eso ya es trabajo de un equipo técnico. En este módulo **no montas un servicio en producción**: dejas el modelo listo y entiendes el panorama.

**Dato real:** cantidad de artículos, tamaño del proveedor (`tamano_num`) y monto total de compras de alimentos de ChileCompra.

**Entregable:** que las **4 celdas de chequeo** muestren ✅.

## Dos formas de usar un modelo desplegado

- **En línea (uno a uno):** llega **un** caso nuevo y se necesita la respuesta al instante (por
  ejemplo, un formulario que, al enviarlo, muestra una estimación). El corazón es una **función de
  predicción**.
- **Por lotes (*batch*):** llega una **tabla** con miles de casos (por ejemplo, al cierre del mes) y se
  necesita estimar el monto de todas de un viaje. El resultado suele ser un archivo CSV exportable.


In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os
import urllib.request
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

# Si los archivos no existen localmente (ej: en Colab), intentamos descargarlos desde GitHub
for archivo in ["compras_ml.csv", "compras_nuevas.csv"]:
    if not os.path.exists(archivo):
        try:
            url = f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/B6-despliegue-de-modelos/{archivo}"  # Reemplazar por la URL real al publicar
            urllib.request.urlretrieve(url, archivo)
            print(f"{archivo} descargado automáticamente.")
        except Exception:
            print(f"Aviso: No se pudo descargar {archivo} automáticamente. Si estás en Colab, asegúrate de subirlo manualmente.")

df = pd.read_csv("compras_ml.csv")
compras_nuevas = pd.read_csv("compras_nuevas.csv")

X = df[["cantidad", "tamano_num"]]
y = df["monto_total"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Recreamos el pipeline de D12
modelo = Pipeline([
    ("escalador", StandardScaler()),
    ("modelo", KNeighborsRegressor(n_neighbors=3))
])
modelo.fit(X_train, y_train)
print(f"Modelo cargado y entrenado exitosamente.")


## 1. La función de predicción: el corazón del despliegue

Casi todo despliegue se reduce a una función que recibe datos nuevos y devuelve una predicción.
Como guardamos el **pipeline completo**, esta función no necesita escalar nada a mano: el pipeline
ya sabe hacerlo.

```python
def predecir_monto(cantidad, tamano_num):
    entrada = pd.DataFrame({"cantidad": [cantidad], "tamano_num": [tamano_num]})
    return modelo.predict(entrada)[0]
```


### ✍️ Ejercicio 1 — Tu función de predicción
Completa `predecir_monto(cantidad, tamano_num)` para que arme la tabla de entrada, llame a
`modelo.predict(...)`, tome el elemento `[0]` y lo devuelva redondeado a 1 decimal.
Completa la celda.


In [ ]:
def predecir_monto(cantidad, tamano_num):
    # TODO: arma un DataFrame con columnas cantidad y tamano_num a partir de los
    #       argumentos, pasalo a modelo.predict(...), toma el elemento [0] y
    #       redondealo a 1 decimal con round(float(...), 1). Devuelvelo.
    return 0.0   # <-- PLACEHOLDER

print("Monto estimado (cant 100, tamano 2):", predecir_monto(100, 2), "CLP")


In [ ]:
# ── Celda de chequeo · Ejercicio 1 ───────────────────────────────────────────
import pandas as pd
try:
    esperado = round(float(modelo.predict(pd.DataFrame({"cantidad": [100], "tamano_num": [2]}))[0]), 1)
    obtenido = predecir_monto(100, 2)
    assert abs(float(obtenido) - esperado) < 0.05, f"predecir_monto(100, 2) deberia dar ~{esperado} CLP."
    print(f"✅ Correcto: tu funcion predecir_monto funciona perfectamente (~{obtenido} CLP).")
except Exception as e:
    print("❌ Aun no. Revisa la creacion del DataFrame o el redondeo.")
    print("   Pista:", e)


## 2. Puntuación por lotes

Para muchos casos a la vez, no llamamos a la función una por una: le pasamos la **tabla entera** al
modelo y obtenemos todas las predicciones de un viaje. Es rápido y es el flujo típico del Estado:
"tengo esta planilla de compras nuevas, devuélvemela con la estimación en una columna".


### ✍️ Ejercicio 2 — Puntuar un lote
Completa `predecir_lote(tabla)` para que agregue una columna `monto_estimado` con la predicción del
modelo para cada fila, y aplícala a `compras_nuevas`. Completa la celda.


In [ ]:
def predecir_lote(tabla):
    resultado = tabla.copy()
    # TODO: agrega la columna 'monto_estimado' con la prediccion del modelo sobre las
    #       columnas cantidad y tamano_num de 'tabla' (redondea con .round(1)).
    resultado["monto_estimado"] = 0.0   # <-- PLACEHOLDER
    return resultado

compras_predichas = predecir_lote(compras_nuevas)
compras_predichas


In [ ]:
# ── Celda de chequeo · Ejercicio 2 ───────────────────────────────────────────
import numpy as np
try:
    assert "monto_estimado" in compras_predichas.columns, "Falta la columna monto_estimado."
    assert len(compras_predichas) == len(compras_nuevas), "Debe haber una prediccion por compra nueva."
    esp = modelo.predict(compras_nuevas[["cantidad", "tamano_num"]]).round(1)
    assert np.allclose(compras_predichas["monto_estimado"].values, esp, atol=1e-6), \
        "Las predicciones de la columna monto_estimado no coinciden con las del modelo."
    print("✅ Correcto: tu funcion predecir_lote funciona correctamente y anadio la columna monto_estimado.")
except Exception as e:
    print("❌ Aun no. Revisa tu funcion predecir_lote.")
    print("   Pista:", e)


## 3. Un despliegue robusto valida la entrada

En el mundo real, a un modelo le llegan datos **malos**: una cantidad imposible (como 10,000 unidades o menor a 0),
un campo vacío, un número donde iba texto. Un buen despliegue **no se cae**: revisa la entrada y responde con un
mensaje claro. Aquí devolveremos un **diccionario** (la forma en que conversan las APIs web).


### ✍️ Ejercicio 3 — Validar la entrada
Completa `predecir_seguro(cantidad, tamano_num)`: si la cantidad no está entre 1 y 500, o el `tamano_num`
no está entre 1 y 4, devuelve `{"ok": False, "error": "Valores fuera de rango"}`; si está bien,
devuelve `{"ok": True, "monto_estimado": predecir_monto(cantidad, tamano_num)}`. Completa la celda.


In [ ]:
def predecir_seguro(cantidad, tamano_num):
    # TODO: si la cantidad NO esta entre 1 y 500, o tamano_num NO esta entre 1 y 4,
    #       devuelve {"ok": False, "error": "..."}. Si esta bien, devuelve
    #       {"ok": True, "monto_estimado": predecir_monto(cantidad, tamano_num)}.
    return {"ok": True, "monto_estimado": 0.0}   # <-- PLACEHOLDER (no valida nada)

print(predecir_seguro(100, 2))
print(predecir_seguro(800, 2))   # cantidad imposible (> 500)


In [ ]:
# ── Celda de chequeo · Ejercicio 3 ───────────────────────────────────────────
try:
    buena = predecir_seguro(100, 2)
    mala  = predecir_seguro(800, 2)
    assert isinstance(buena, dict) and isinstance(mala, dict), "La funcion debe devolver un diccionario."
    assert mala.get("ok") == False, "Para una cantidad fuera de rango (800), 'ok' debe ser False."
    assert buena.get("ok") == True and abs(float(buena.get("monto_estimado")) - predecir_monto(100, 2)) < 0.1, \
        "Para valores correctos, debe devolver {'ok': True, 'monto_estimado': ...}."
    print("✅ Correcto: tu funcion predecir_seguro maneja errores y valida rangos correctamente.")
except Exception as e:
    print("❌ Aun no. Revisa tus validaciones de rango y la estructura del diccionario devuelto.")
    print("   Pista:", e)


## 4. Entregar el resultado

El producto final de una puntuación por lotes suele ser un **archivo** que se comparte. Guardamos la
tabla con las predicciones en un CSV, listo para enviar a un colega o subir a un sistema.

```python
compras_predichas.to_csv("predicciones_monto.csv", index=False)
```


### ✍️ Ejercicio 4 — Generar el archivo de predicciones
Guarda `compras_predichas` en un archivo `predicciones_monto.csv` (sin el índice). Completa
la celda.


In [ ]:
# TODO: guarda el resultado del lote (compras_predichas) en un archivo CSV
#       llamado "predicciones_monto.csv", sin el indice:
#       compras_predichas.to_csv("predicciones_monto.csv", index=False)

import os
print("Archivo creado:", os.path.exists("predicciones_monto.csv"))


In [ ]:
# ── Celda de chequeo · Ejercicio 4 ───────────────────────────────────────────
import os, pandas as pd
try:
    assert os.path.exists("predicciones_monto.csv"), "No encuentro el archivo predicciones_monto.csv."
    leido = pd.read_csv("predicciones_monto.csv")
    assert "monto_estimado" in leido.columns, "El CSV debe incluir la columna monto_estimado."
    assert len(leido) == len(compras_predichas), "La longitud del CSV debe coincidir con compras_predichas."
    print("✅ Correcto: guardaste las predicciones en el archivo CSV sin el indice. ¡Despliegue superado!")
except Exception as e:
    print("❌ Aun no. Revisa la exportacion a CSV.")
    print("   Pista:", e)


## 5. Más allá: convertirlo en un servicio (y cuidarlo)

La función que escribiste es **exactamente** lo que envuelven las formas más avanzadas de
despliegue. No las ejecutamos aquí (requieren un servidor), pero conviene conocerlas:

- **Una API web** (con `Flask` o `FastAPI`): expone tu función en una dirección de internet para que
  otros sistemas la consulten. En esencia, es tu `predecir_seguro` detrás de una URL:

  ```python
  # (ilustrativo, no se ejecuta en este cuaderno)
  from flask import Flask, request, jsonify
  app = Flask(__name__)

  @app.route("/predecir")
  def predecir():
      cant = float(request.args["cantidad"]); tam = float(request.args["tamano_num"])
      return jsonify(predecir_seguro(cant, tam))
  ```

- **Una app web simple** (con `Gradio`): crea, con pocas líneas, una página con casillas donde
  cualquiera escribe la cantidad y el tamaño del proveedor y ve la predicción. Ideal para que personas no técnicas
  usen el modelo.

### Y lo más importante: cuidar el modelo en el tiempo
Un modelo **no es para siempre**. El mundo cambia (el clima, la población, los procesos), y un
modelo entrenado con datos viejos se va **desajustando**. Por eso un despliegue serio incluye:

- **Monitoreo:** vigilar si las predicciones siguen siendo buenas comparándolas con la realidad.
- **Reentrenamiento:** volver a entrenar con datos frescos cada cierto tiempo.
- **Transparencia:** documentar qué datos y supuestos usa, sus límites y quién responde por él
  (esto enlaza con la ética de datos de M7).


## 6. Cierre — ¡completaste la Línea B! 🎓

Desde D8 recorriste el camino entero de la ciencia de datos aplicada al Estado:

- **D8** fabricaste *features* con SQL.
- **D9** entrenaste tu primer modelo de regresión con compras públicas reales.
- **D10** dominaste los árboles de decisión y bosques aleatorios y aprendiste a controlar el sobreajuste.
- **D11** clasificaste proveedores por tamaño y agrupaste compras similares con clustering.
- **D12** encadenaste los pasos en pipelines reproducibles y los guardaste con `joblib`.
- **D13** expusiste tu pipeline mediante funciones de predicción seguras y automatizaste el scoring por lotes.

Ya no eres solo un consumidor de datos: eres un constructor de soluciones predictivas preparadas para
el servicio público. ¡Felicitaciones! 🎓

### Mini-glosario
- **Puntuación en línea:** estimar un caso nuevo al instante a través de un servicio.
- **Puntuación por lotes (*batch*):** estimar el valor de toda una base de datos o planilla de un viaje.
- **Validación de entrada:** comprobar que los datos recibidos están en rangos válidos antes de estimar,
  para evitar que el servidor falle.

---
*Fuente de datos: Dirección de Compras y Contratación Pública (ChileCompra / MercadoPúblico).*
*Contenido bajo licencia CC BY 4.0 · Formación Pública.*
